# How prompt framing changes a design decision

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmudrc/design-research-workshop/blob/v0.1.0/notebooks/laptop_prompt_study.ipynb)

Compare a balanced student laptop design with one that prioritizes demand. A deterministic workshop proxy proposes configurations, and the packaged evaluator scores them.

In [ ]:
%pip install "design-research-workshop @ git+https://github.com/cmudrc/design-research-workshop.git@v0.1.0"

In [ ]:
from pathlib import Path
from statistics import mean

import design_research as dr
from design_research_workshop import PROBLEM_ID, laptop_study_agent, workshop_client

price_limit = 1000  # Try 900 and compare the scores again.
task = f"Choose a student laptop configuration as JSON with a price ceiling of ${price_limit:,}."
prompts = {
    "balanced": f"{task} Choose a balanced design.",
    "demand_focused": f"{task} Prioritize expected demand.",
}
problem = dr.problems.get_problem(PROBLEM_ID)
print(f"Catalog size: {problem.candidate_count:,} configurations")

In [ ]:
output_dir = Path("artifacts/laptop_prompt_study")
study = dr.experiments.build_strategy_comparison_study(
    dr.experiments.StrategyComparisonConfig(
        study_id="workshop_laptop_prompt_study",
        title="Student laptop prompt comparison",
        description="Compare balanced and demand-focused proxy responses.",
        bundle=dr.experiments.BenchmarkBundle(
            bundle_id="workshop_laptop",
            name="Student laptop comparison",
            description="Two prompt conditions on one packaged decision problem.",
            problem_ids=(PROBLEM_ID,),
            agent_specs=tuple(prompts),
        ),
        run_budget=dr.experiments.RunBudget(replicates=2, parallelism=1),
        output_dir=output_dir,
    )
)
conditions = dr.experiments.build_design(study)

with workshop_client() as client:
    bindings = {
        name: laptop_study_agent(client, prompt)
        for name, prompt in prompts.items()
    }
    results = dr.experiments.run_study(
        study, conditions=conditions, agent_bindings=bindings,
        checkpoint=False, show_progress=False,
    )

In [ ]:
paths = dr.experiments.export_analysis_tables(
    study, conditions=conditions, run_results=results, output_dir=output_dir,
)
validation = dr.analysis.validate_experiment_events(paths["events.csv"])
assert validation.is_valid, validation.errors

rows = dr.analysis.build_condition_metric_table_from_artifacts(
    paths["events.csv"], metric="predicted_share", condition_column="agent_id",
)
assert len(rows) == len(results), "Each run should produce a scored design."
for name in prompts:
    scores = [float(row["value"]) for row in rows if row["condition"] == name]
    print(f"{name}: {mean(scores):.1%} mean predicted market share")
print(f"Validated {validation.n_rows} event rows in {paths['events.csv']}")

Lower the price ceiling to $900 and rerun from the parameter cell. Which condition changes score? Repeated runs of a given prompt use the same rule-based policy. The outputs illustrate the workflow and its evaluator.